# Solutions · Chapter 05-07 · Capacity

Worked answers to every exercise in `notebooks/05_regression/05-07_capacity.ipynb`.

In [ ]:
import warnings
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore")

# SYNTHETIC: the chapter's 600 shop-days.
# TRUTH: 200 + 0.9 footfall + 15 promo + 40 weekend + 120 (promo AND weekend), noise sd 25.
shop_rng = np.random.default_rng(9)
n_days = 600
footfall = shop_rng.uniform(50, 400, n_days)
promo = shop_rng.integers(0, 2, n_days).astype(float)
weekend = shop_rng.integers(0, 2, n_days).astype(float)
sales = (200 + 0.9 * footfall + 15 * promo + 40 * weekend
         + 120 * promo * weekend + shop_rng.normal(0, 25, n_days))
shop = pd.DataFrame({"footfall": footfall, "promo": promo, "weekend": weekend})

# SYNTHETIC: the chapter's cubic. TRUTH: 2 + 1.5x - 0.8x^2 + 0.15x^3, noise sd 3.0.
curve_rng = np.random.default_rng(4)
n_points = 120
position = curve_rng.uniform(-4, 4, n_points)
outcome = (2.0 + 1.5 * position - 0.8 * position ** 2 + 0.15 * position ** 3
           + curve_rng.normal(0, 3.0, n_points))
fit_x, held_x, fit_y, held_y = train_test_split(
    position, outcome, test_size=0.5, random_state=0)


def polynomial_fit(degree, train_x, train_y):
    model = make_pipeline(PolynomialFeatures(degree, include_bias=False),
                          StandardScaler(), LinearRegression())
    return model.fit(train_x.reshape(-1, 1), train_y)


print("set up: %d shop-days and %d curve points" % (n_days, n_points))

## Quick understanding

### E1 · Capacity

**Capacity is the size of the set of functions a model can produce.** A straight line can be any line;
add a squared term and it can be any parabola, a set that contains every line, so capacity has strictly
grown.

Two things that increase it in a linear model:

1. **More columns** - another feature, a polynomial term, an interaction, a one-hot expansion of a
   categorical variable. Each one is a parameter the fit can choose.
2. **A richer basis for the same feature** - splines, or a higher polynomial degree, which buys shapes
   rather than variables.

The count of columns is the workable proxy, and the ratio that matters is columns to rows.

### E2 · The two signatures

| | training error | gap to held-out |
|---|---|---|
| **Underfitting** | high, well above the noise floor | small, sometimes negative |
| **Overfitting** | low, often below the noise floor | large |

**Both parts are needed.** A small gap alone is not good news - degree 1 had a gap of -0.45 and was the
worst model in the chapter. It is the *pair* that diagnoses.

### E3 · Why the training residual plot misses overfitting

**Because overfitting is the model fitting those exact rows** - so on those rows it looks like success.
Degree 18 had the tightest training residuals of the three models in the chapter and the worst held-out
RMSE.

Underfitting is different in kind: it is a failure to represent the truth, and the unrepresented part
stays in the residuals wherever you look. **Underfitting is visible on training rows. Overfitting is not,
by construction** - which is exactly what a held-out split buys.

## Hand calculation

### E4 · Is there an interaction?

| | no promo | promo | lift |
|---|---|---|---|
| weekday | 100 | 130 | **+30** |
| weekend | 150 | 180 | **+30** |

**No interaction.** The promotion is worth 30 in both rows, so one coefficient describes it.

The arithmetic is the **difference in differences**: `(180 - 150) - (130 - 100) = 30 - 30 = 0`. That
single number is the interaction, and zero means the lines are parallel.

Note that both variables still matter - the weekend is worth +50 and the promotion +30. **No interaction
does not mean no effect;** it means the effects add.

### E5 · Now weekend/promo is 240

Difference in differences: `(240 - 150) - (130 - 100) = 90 - 30 = ` **+60**.

Reading the coefficients straight off the table, since with two binary variables the four cells and the
four coefficients carry identical information:

- **intercept** = the cell where both are 0 = **100**
- **promo** = weekday lift = `130 - 100` = **30**
- **weekend** = no-promo lift = `150 - 100` = **50**
- **promo x weekend** = the difference in differences = `240 - 100 - 30 - 50` = **60**

Check: weekend with promo = `100 + 30 + 50 + 60 = 240`. ✓

**And the reading that matters:** the promotion is worth 30 on a weekday and `30 + 60 = 90` at the
weekend. The `promo` coefficient of 30 is *not* the effect of promotions in general.

### E6 · Columns from a degree-2 expansion of 6 features

`C(6 + 2, 2) - 1 = 28 - 1 = ` **27 columns**, in three kinds:

| kind | count | why |
|---|---|---|
| the original features | 6 | `a, b, c, d, e, f` |
| squares | 6 | `a², b², ...` |
| pairwise products | 15 | `C(6, 2)` |

6 + 6 + 15 = 27. ✓ The pairwise products are the majority already at degree 2, and they are the term that
grows fastest as features are added.

### E7 · Training RMSE 2.0, held-out 2.1, noise sd 5.0

**The gap is tiny and both errors are far below the noise floor, which is impossible.**

A model cannot predict better than the data's own irreducible noise. RMSE 2.0 against a noise standard
deviation of 5.0 is not a good model - it is evidence that one of the premises is false:

1. **Leakage** (04-05) - a feature that contains the answer. By far the most likely.
2. **The noise estimate of 5.0 is wrong**, which happens when it comes from a different population or a
   different unit of observation.
3. **Duplicate rows across the split**, so "held-out" rows were also trained on - 04-04's grouped split
   problem.

**What is odd is precisely that nothing looks odd.** Neither of this chapter's two failure modes is
present, and a practitioner reading only "small gap, low error" would ship it. **A score better than the
noise floor is a red flag, not a triumph.**

## Coding

### E8 · The non-parallel-lines diagnostic

In [ ]:
def interaction_plot(frame, first, second, target, ax=None, adjust=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    values = frame[target] if adjust is None else frame[target] - adjust
    means = pd.DataFrame({first: frame[first], second: frame[second], "value": values})
    cells = means.groupby([second, first], observed=True)["value"].mean().unstack()
    for level in cells.index:
        ax.plot(cells.columns.to_numpy(), cells.loc[level].to_numpy(), "o-",
                linewidth=2.4, markersize=9, label="%s = %g" % (second, level))
    ax.set_xticks(cells.columns.to_numpy())
    ax.set_xlabel(first)
    ax.set_ylabel("mean %s" % target)
    ax.legend(fontsize=9)
    lifts = cells.iloc[:, -1] - cells.iloc[:, 0]
    return cells, lifts


with_busy = shop.assign(
    sales=sales,
    busy=(shop.footfall > shop.footfall.median()).astype(float))

fig, (left, right) = plt.subplots(1, 2, figsize=(12.8, 4.4), sharey=True)
footfall_effect = 0.9 * with_busy.footfall

cells_a, lifts_a = interaction_plot(with_busy, "promo", "weekend", "sales", left,
                                    adjust=footfall_effect)
left.set_title("promo x weekend: lifts %.1f and %.1f" % tuple(lifts_a), fontsize=11)

cells_b, lifts_b = interaction_plot(with_busy, "promo", "busy", "sales", right,
                                    adjust=footfall_effect)
right.set_title("promo x busy: lifts %.1f and %.1f" % tuple(lifts_b), fontsize=11)

plt.tight_layout()
plt.show()

print("promo x weekend lifts:", np.round(lifts_a.to_numpy(), 1))
print("promo x busy    lifts:", np.round(lifts_b.to_numpy(), 1))

**One pair interacts and the other does not, and the plot says so at a glance.**

- **`promo` x `weekend`:** lifts of **17.6 and 141.1**. The lines are wildly non-parallel. This is the
  interaction the data was built with.
- **`promo` x `busy`:** lifts of **74.0 and 85.0**. Nearly parallel, and the 11-point difference is
  small next to the noise standard deviation of 25 - so there is nothing here worth a column.

**Two details in the function are worth copying.**

The `adjust` argument subtracts a continuous feature's known effect before averaging. Without it, any
imbalance in `footfall` across the four cells would show up as a fake interaction - the group means would
be comparing cells that differ in something else as well. **This is 05-05's "two things changed at once"
in a new place**, and it is why the chapter's table subtracts `0.9 * footfall` first.

Returning the lifts alongside the plot matters because "non-parallel" is a judgement about *size*.
Lines are never exactly parallel on real data; the question is whether the gap is large next to the
noise, and printing the two numbers is what turns the picture into a decision.

### E9 · Expansions against a hand-picked term

In [ ]:
train_shop, test_shop, train_sales, test_sales = train_test_split(
    shop, sales, test_size=0.3, random_state=0)

results = []
for degree in [1, 2, 3]:
    expander = PolynomialFeatures(degree, include_bias=False)
    model = make_pipeline(expander, StandardScaler(), LinearRegression())
    model.fit(train_shop, train_sales)
    prediction = model.predict(test_shop)
    results.append({"model": "full expansion, degree %d" % degree,
                    "columns": expander.fit(train_shop).n_output_features_,
                    "test R2": r2_score(test_sales, prediction),
                    "test RMSE": np.sqrt(((test_sales - prediction) ** 2).mean())})

hand_train = train_shop.assign(promo_x_weekend=train_shop.promo * train_shop.weekend)
hand_test = test_shop.assign(promo_x_weekend=test_shop.promo * test_shop.weekend)
hand_model = LinearRegression().fit(hand_train, train_sales)
hand_prediction = hand_model.predict(hand_test)
results.append({"model": "hand-picked interaction", "columns": hand_train.shape[1],
                "test R2": r2_score(test_sales, hand_prediction),
                "test RMSE": np.sqrt(((test_sales - hand_prediction) ** 2).mean())})

print(pd.DataFrame(results).to_string(index=False, float_format=lambda v: "%.4f" % v))

**I would ship the hand-picked model, and the table makes the case without any appeal to taste.**

It has **4 columns and the best test RMSE of the four (23.412)**. The degree-3 expansion needs **19
columns** to arrive at 23.480 - slightly worse, with nearly five times the parameters.

Three reasons beyond the score:

1. **It is readable.** Four coefficients, each of which someone can check against what they know about
   the shop. The degree-3 model has a `footfall^2 * promo` term that nobody can defend.
2. **It will hold up better off this sample.** The extra fifteen columns are fitting noise; here there
   are 420 training rows so the cost is small, and on a slice with 60 rows it would not be.
3. **It says what it found.** "Promotions are worth 21 midweek and 143 at the weekend" is the deliverable.
   The expansion contains that fact and does not report it.

**The honest caveat:** the full expansion found essentially the same answer with no domain knowledge at
all, going from 0.8645 to 0.9541 on its own. If nobody had known about weekends, degree 2 would have
rescued the model - which is the argument for trying an expansion when a model disappoints, and then
going to look at what it used.

### E10 · How stable is the chosen degree?

In [ ]:
degrees = list(range(1, 13))


def choose_by_cross_validation(seed):
    scores = []
    for degree in degrees:
        pipeline = make_pipeline(PolynomialFeatures(degree, include_bias=False),
                                 StandardScaler(), LinearRegression())
        folds = KFold(5, shuffle=True, random_state=seed)
        scores.append(-cross_val_score(pipeline, position.reshape(-1, 1), outcome,
                                       cv=folds,
                                       scoring="neg_root_mean_squared_error").mean())
    return degrees[int(np.argmin(scores))]


def choose_by_single_split(seed):
    train_x, test_x, train_y, test_y = train_test_split(
        position, outcome, test_size=0.5, random_state=seed)
    best, best_error = None, np.inf
    for degree in degrees:
        model = polynomial_fit(degree, train_x, train_y)
        error = np.sqrt(((test_y - model.predict(test_x.reshape(-1, 1))) ** 2).mean())
        if error < best_error:
            best, best_error = degree, error
    return best


cv_picks = [choose_by_cross_validation(seed) for seed in range(10)]
split_picks = [choose_by_single_split(seed) for seed in range(10)]

print("5-fold cross-validation, 10 seeds:", cv_picks)
print("   counts:", pd.Series(cv_picks).value_counts().sort_index().to_dict())
print("\none 50/50 split, 10 seeds     :", split_picks)
print("   counts:", pd.Series(split_picks).value_counts().sort_index().to_dict())
print("\nthe data was generated at degree 3")

**Cross-validation picks degree 3 in 9 runs out of 10. A single split picks it in 6, and otherwise
wanders to 4, 6, 6 and 9.**

**This is 04-03's split lottery, now choosing the model rather than merely scoring it** - and that is the
more expensive version of the problem. A bad score estimate is a number you can caveat; a bad *choice*
ships a degree-9 polynomial.

The mechanism is the one 04-03 gave. A single 50/50 split scores on 60 rows, and the held-out curve in
the chapter was almost flat between degrees 3 and 11 - RMSE 3.26 to 3.50 - so noise of a tenth of an RMSE
unit is enough to move the winner. Cross-validation averages five such estimates and the noise falls.

**The practical rule: never select a hyperparameter on a single split if you can afford folds.** And when
the curve near the optimum is flat, prefer the *simplest* setting within noise of the best rather than
the argmin - here that is degree 3 every time.

### E11 · Does centring help?

In [ ]:
def condition_of(matrix):
    with_intercept = np.column_stack([np.ones(len(matrix)), matrix])
    return float(np.linalg.cond(with_intercept.T @ with_intercept / len(matrix)))


def expand(values, degree):
    return PolynomialFeatures(degree, include_bias=False).fit_transform(
        values.reshape(-1, 1))


as_a_year = fit_x + 2005.0        # the same numbers, shifted to look like a year

rows = []
for degree in [3, 8, 12]:
    for label, values in [("x, already near zero", fit_x), ("x + 2005, a 'year'", as_a_year)]:
        raw = condition_of(expand(values, degree))
        centred = condition_of(expand(values - values.mean(), degree))
        rows.append({"column": label, "degree": degree, "raw": raw, "centred": centred,
                     "improvement": raw / centred})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: "%.4g" % v))

**Centring is worth nothing on this chapter's `x` and worth 8.7e+24 on the same numbers shifted by 2005.**

`fit_x` has a mean of about 0.23 already, so subtracting it changes almost nothing - the improvements are
**1.03x, 0.97x and 0.59x**, and at degrees 8 and 12 centring made the conditioning slightly *worse*.

Shift the identical values to sit around 2005 and the raw degree-3 expansion has a condition number of
**8.5e+27**; at degree 12 it is **4.8e+74**. Double precision holds about 16 digits, so those fits have no
meaningful digits at all. Centring returns them to **976.9** and **1.27e+15** - the exact values the
unshifted column already had, which is the point: **after centring, the two columns are the same data,
and the shift was pure damage.**

**The rule, stated properly:** centring matters exactly to the extent that the feature's mean is large
relative to its spread. Here that ratio is 0.1 and centring is a no-op. For a `year`, a `timestamp`, a
`postcode` or a UNIX epoch it is enormous, and **expanding such a column to any power without centring
first is one of the most reliable ways to destroy a fit.**

**And centring is not a substitute for scaling.** Even centred, degree 12 sits at 1.27e+15 - the chapter's
`StandardScaler` after expansion is what brings it to 1.6e+08. Do both.

### E12 · When the interaction is the only signal

In [ ]:
# SYNTHETIC: 800 rows where NEITHER main effect exists.
# TRUTH: 50 - 20a - 20b + 40ab, so the mean of y is the same for a=0 and a=1.
xor_rng = np.random.default_rng(31)
n_xor = 800
first = xor_rng.integers(0, 2, n_xor).astype(float)
second = xor_rng.integers(0, 2, n_xor).astype(float)
xor_y = (50 - 20 * first - 20 * second + 40 * first * second
         + xor_rng.normal(0, 5, n_xor))
xor = pd.DataFrame({"a": first, "b": second})

print("mean of y by a:", pd.Series(xor_y).groupby(first).mean().round(2).to_dict())
print("mean of y by b:", pd.Series(xor_y).groupby(second).mean().round(2).to_dict())
print("\nmean of y in each cell:")
print(pd.DataFrame({"a": first, "b": second, "y": xor_y})
      .groupby(["a", "b"])["y"].mean().round(2).unstack().to_string())

xor_train, xor_test, y_train, y_test = train_test_split(
    xor, xor_y, test_size=0.3, random_state=0)
plain = LinearRegression().fit(xor_train, y_train)
richer_train = xor_train.assign(a_x_b=xor_train.a * xor_train.b)
richer_test = xor_test.assign(a_x_b=xor_test.a * xor_test.b)
richer = LinearRegression().fit(richer_train, y_train)

print("\nmain effects only: test R2 %+.4f   coefficients a %+.3f, b %+.3f"
      % (r2_score(y_test, plain.predict(xor_test)), plain.coef_[0], plain.coef_[1]))
print("with a x b       : test R2 %+.4f   coefficients %s"
      % (r2_score(y_test, richer.predict(richer_test)),
         {name: round(float(value), 3)
          for name, value in zip(richer_train.columns, richer.coef_)}))

**The main-effects model scores -0.0052 - worse than predicting the mean - and reports coefficients of
+2.07 and +0.26, both indistinguishable from zero. Adding one column takes it to 0.7644.**

The marginal means say why: **38.86 against 40.49** for `a`, and 39.65 against 39.73 for `b`. Averaged
over the other variable, neither one moves the target at all. The cell means show where the signal
actually lives:

| | b = 0 | b = 1 |
|---|---|---|
| **a = 0** | 49.44 | 29.70 |
| **a = 1** | 30.66 | 49.76 |

The two variables *agreeing* is what matters, and no additive model can express "agreement".

**Three consequences worth carrying.**

**Feature selection by univariate screening would delete both columns.** Any procedure that ranks
features by their individual correlation with the target - a common first step - scores both at
approximately zero and drops them, discarding 76% of the explainable variance.

**A coefficient near zero does not mean a variable does not matter.** It means it does not matter *on
average, holding nothing else in mind*, which is a much weaker statement than it sounds.

**This is the case for trying a tree-based model early.** 05-10's trees find this split without being
told to look, because splitting is inherently conditional. A large gap between a linear model and a
boosted one, on the same columns, is evidence of exactly this shape.

## Interpretation

### E13 · Training 0.99, held-out 0.35, "collect more data"

**More data will help, and it is the wrong thing to try first** because it is the most expensive item on
the list.

A gap of 0.64 is overfitting, and the chapter established that more rows genuinely fix overfitting -
degree 16 went from -3.107 on 18 rows to +0.881 on 1,800. So the instinct is right.

**What I would try first, in ascending order of cost:**

1. **Count parameters against rows.** If it is 200 rows and 180 columns the diagnosis is finished, and
   the fix is fewer columns, not more rows.
2. **Reduce capacity.** Drop the polynomial degree, remove features, or use 05-09's regularisation, which
   keeps the columns and constrains their coefficients. Minutes, not weeks.
3. **Check the split.** A held-out score can also collapse because the split is grouped or chronological
   in a way the random split ignored - 04-04 - and that looks identical from the metrics alone. Reshuffle
   and rerun a few times.
4. **Check for leakage in the other direction.** Training R-squared of 0.99 is high enough to be worth a
   look on its own.
5. **Then collect more data**, once you know how much it would buy - which is a learning curve, and
   05-08 is about drawing one.

**The move being modelled:** the proposed fix is not wrong, it is unmeasured. "Would more data help, and
how much?" is answerable in an afternoon by fitting on subsets of what you already have.

### E14 · Held-out score better than training

**Explanation one: underfitting.** A model too stiff to fit the noise cannot fit the training noise
either, so there is no gap to lose. Degree 1 in the chapter had training RMSE 5.61 and held-out 5.16.

**Explanation two: an easier test set, by luck.** With a small held-out set the split lottery can hand
you rows that happen to be less noisy or less extreme. This is 04-03, and it is more likely the smaller
the test set.

**How to tell them apart:** check whether the training error is near the noise floor. Underfitting has a
training error well *above* what the data allows, and split luck does not.

If the noise floor is unknown, **reshuffle the split five times.** Underfitting produces the inversion
every time, because it is a property of the model. Split luck produces it once or twice out of five.

**A third possibility worth naming:** in cross-validation, a fold's training set is larger than the whole
data used to score, so scores computed on differently-sized sets are not comparable in the first place.
Check that you are comparing like with like before diagnosing anything.

## Debugging

### E15 · Degree 4 collapses, coefficients of order 1e+9

**Diagnosis: the features were expanded without being scaled, and the fit is numerical noise.**

Coefficients of 1e+9 are the signature. They arise when the expanded columns are on wildly different
scales - the chapter measured a condition number of 2.7e+09 at degree 8 on raw powers, and 7.5e+14 at
degree 12 - and the solver returns enormous, nearly-cancelling values. Change one row and they change
completely, which is why held-out performance collapses.

**Check it in one line:** `X.std(axis=0)` after expansion. Orders of magnitude between columns is the
confirmation.

**The fix, in order:**

1. **Put a `StandardScaler` after `PolynomialFeatures`** - inside a `Pipeline`, so 04-07's rule about
   fitting the scaler on training folds only is not broken.
2. **Centre first if the column is far from zero.** A raw `year` expanded to degree 3 has a condition
   number of 3.4e+28, as E11 showed.
3. **If the coefficients are still large after scaling, the problem is statistical rather than
   numerical** - that is genuine overfitting, and 05-09's ridge penalty is the tool.

**The distinction matters:** an unscaled expansion is a bug with a one-line fix, and overfitting is a
modelling decision. Both produce large coefficients and a collapsed score.

### E16 · A near-zero `promo` coefficient with an interaction present

**The most likely explanation: the coefficient is being misread.** With `promo x weekend` in the model,
`promo` is the effect of a promotion **when `weekend` is 0** - not in general. If promotions genuinely do
little midweek, a near-zero `promo` coefficient is *correct* and says nothing about weekends.

Two other possibilities:

- **A near-collinear pair.** If promotions almost always run at the weekend, `promo` and
  `promo x weekend` are nearly the same column, and 05-03 showed what that does: the individual
  coefficients become unstable and large with opposite signs while the prediction stays fine. Check with
  a cross-tabulation of the two columns and the VIF.
- **The business is right and the model is missing the mechanism** - promotions may work through a
  channel the model does not have, in which case the effect is in the residuals, not the coefficient.

**What to report instead of a coefficient: the fitted difference.** "Sales are X higher on promotion
weekdays and Y higher on promotion weekends" - two predictions from the model, each with an interval.
That is the quantity anyone actually asked for, it is invariant to how the model was parameterised, and
**E21 shows the two parameterisations give literally identical predictions while their coefficients look
nothing alike.**

## Exam and interview reasoning

### E17 · "How do you know if a model is overfitting?"

> "I compare the error on the rows it was fitted to with the error on rows it has never seen. A large gap
> is overfitting. But I check both numbers, because a small gap with a high error is underfitting, which
> needs the opposite fix - and I look at where the training error sits relative to the noise I think is in
> the data. Then I look at the residuals on the held-out rows, because a metric tells me *that* something
> is wrong and the plot tells me what."

**"And if you only have 200 rows and cannot afford a test set?"**

> "Then I cross-validate rather than hold out - five folds still score every row, just never with a model
> that saw it. If I am also choosing a hyperparameter I would nest it, because 04-07 showed that choosing
> and scoring on the same folds is optimistic. And I would lean harder on the things that need no split
> at all: a low parameter-to-row ratio, and regularisation, which lets me keep the columns without
> spending the capacity."

**What is being tested:** the first question checks that you say "gap" rather than "bad score". The
follow-up checks whether "hold out a test set" is a procedure you have memorised or a principle you can
adapt when the resource it assumes is not there.

## Transfer to a different situation

### E18 · Readmission, 40 features, 8 high-cardinality categoricals

**The categoricals are the whole capacity question, and it is not close.**

One-hot encoding 8 columns that each have, say, 50 levels adds **400 columns** before any interaction is
considered. That is the dominant term - the 32 numeric features are a rounding error next to it - so
"how do I think about capacity here" is mostly "what do I do with those eight columns".

**What I would expand:**

- **Two or three interactions I can name a reason for** - age with comorbidity count, length of stay with
  discharge destination. Domain knowledge, as in the chapter, is the best source and it is free.
- **Non-linearity in a small number of continuous features**, and by binning or splines rather than
  polynomial degree, because clinical variables are rarely polynomial and bins are readable by clinicians.

**What I would not expand:**

- **A full polynomial expansion of 40 features.** Degree 2 alone is 860 columns.
- **Interactions among the one-hot columns.** Two 50-level categoricals interacted is 2,500 columns, most
  of which will have single-digit row counts - the target-encoding trap of 04-06.

**What I would do to the categoricals instead:** group rare levels into "other" with a frequency
threshold; consider target encoding *inside a pipeline* with cross-fitting, since 04-05 showed what
happens otherwise; and check whether a coarser column already exists - service line rather than
individual ward.

**And the honest framing:** with 400 potential columns, this is the case where a model that manages its
own capacity - regularised regression from 05-09, or gradient boosting from 05-11 - is a better default
than hand-picking terms. **The chapter's lesson is not "expand less"; it is "know what you are
spending".**

## Explain it to someone non-technical

### E19 · Overfitting without a curve

> Imagine revising for an exam using last year's paper. One person learns the material; another memorises
> which answers were A, B and C. Both score full marks on that paper, so the practice test cannot tell
> them apart. Give them this year's paper and only one of them still does well.
>
> That is what we are guarding against. A model that can memorise will always look perfect on the data we
> showed it, so the only honest test is data it has never seen - which is why we deliberately hide some
> before we start.

*(87 words.)* The analogy earns its place because it also explains **why the held-out set has to be
hidden from the start**, which is the operational point a manager needs, and it makes leakage easy to
describe later - someone left this year's answers in the revision folder.

## Optional challenge

### E20 · Where does degree 16 stop being worse than degree 3?

In [ ]:
def average_rmse_at(total_rows, repeats=12):
    collected = {3: [], 16: []}
    for repeat in range(repeats):
        local = np.random.default_rng(1000 + repeat)
        x_here = local.uniform(-4, 4, total_rows)
        y_here = (2.0 + 1.5 * x_here - 0.8 * x_here ** 2 + 0.15 * x_here ** 3
                  + local.normal(0, 3.0, total_rows))
        train_x, test_x, train_y, test_y = train_test_split(
            x_here, y_here, test_size=0.4, random_state=repeat)
        for degree in (3, 16):
            model = polynomial_fit(degree, train_x, train_y)
            collected[degree].append(
                np.sqrt(((test_y - model.predict(test_x.reshape(-1, 1))) ** 2).mean()))
    return len(train_x), float(np.mean(collected[3])), float(np.mean(collected[16]))


sizes = [40, 60, 100, 200, 400, 800, 1600, 3200]
crossover = []
for total in sizes:
    train_count, three, sixteen = average_rmse_at(total)
    crossover.append({"rows used to fit": train_count, "RMSE at degree 3": three,
                      "RMSE at degree 16": sixteen, "ratio": sixteen / three,
                      "rows per parameter": train_count / 16})
crossover = pd.DataFrame(crossover)
print(crossover.to_string(index=False, float_format=lambda v: "%.3f" % v))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

left.plot(crossover["rows used to fit"], crossover["ratio"], "o-", color="#7B3294",
          linewidth=2.4, markersize=9)
left.axhline(1.0, color="#009E73", linewidth=2, linestyle="--",
             label="degree 16 as good as degree 3")
left.set_xscale("log")
left.set_yscale("log")
left.set_xlabel("rows used to fit (log scale)")
left.set_ylabel("degree 16 RMSE / degree 3 RMSE")
left.set_title("Three orders of magnitude of damage, gone", fontsize=11)
left.legend(fontsize=9)

tail = crossover[crossover["rows used to fit"] >= 120]
right.plot(tail["rows used to fit"], tail["ratio"], "o-", color="#7B3294",
           linewidth=2.4, markersize=10)
for rows_used, ratio in zip(tail["rows used to fit"], tail["ratio"]):
    right.annotate("%.3f" % ratio, (rows_used, ratio), textcoords="offset points",
                   xytext=(0, 11), ha="center", fontsize=9.5)
right.axhline(1.0, color="#009E73", linewidth=2, linestyle="--",
              label="no cost at all")
right.axhline(1.05, color="#E69F00", linewidth=1.8, linestyle=":", label="within 5%")
right.set_xscale("log")
right.set_ylim(0.99, 1.12)
right.set_xlabel("rows used to fit (log scale)")
right.set_ylabel("degree 16 RMSE / degree 3 RMSE")
right.set_title("The tail: within 5% at 240 rows, and never quite free", fontsize=11)
right.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The ratio falls 995 -> 41.8 -> 3.55 -> 1.70 -> 1.033 -> 1.016 -> 1.008 -> 1.005.**

Degree 16 is **within 5% of degree 3 at 240 fitting rows**, and within 1% by about 960.

**The shape is the interesting part, and it has two regimes.**

Below about 60 rows the ratio is not just large but *wild* - 995 at 24 rows. With 16 parameters and 24
rows the fit is nearly unconstrained, and the RMSE is dominated by whatever the most extreme
extrapolation happens to be on that draw. This is the region where a single bad fit moves the average by
orders of magnitude, which is why the numbers are averaged over twelve repeats.

Above about 240 rows the ratio decays smoothly towards 1 and never quite reaches it. **The extra
parameters never become free - they become negligible**, costing about half a percent at 1,920 rows.

**The last column converts this into the rule of thumb the chapter gestured at.** 240 rows for 16
parameters is **15 rows per parameter** for "within 5%", and 60 per parameter for "within 1%". Those
figures are specific to this generator, this noise level and this measure of "close enough" - a smoother
truth or less noise would move them - but the order of magnitude is the useful thing, and it is a far
better instinct than any fixed rule about degree.

### E21 · Two parameterisations, one model

In [ ]:
interaction_design = pd.DataFrame({
    "footfall": footfall, "promo": promo, "weekend": weekend,
    "promo_x_weekend": promo * weekend})

cell_index = (promo * 2 + weekend).astype(int)      # 0, 1, 2, 3 for the four cells
indicator_design = pd.DataFrame({"footfall": footfall})
for level in [1, 2, 3]:                              # cell 0 is the reference
    indicator_design["cell_%d" % level] = (cell_index == level).astype(float)

interaction_model = LinearRegression().fit(interaction_design, sales)
indicator_model = LinearRegression().fit(indicator_design, sales)

interaction_prediction = interaction_model.predict(interaction_design)
indicator_prediction = indicator_model.predict(indicator_design)

print("largest difference between the two sets of predictions: %.3e"
      % np.abs(interaction_prediction - indicator_prediction).max())
print("R-squared: %.10f and %.10f\n"
      % (interaction_model.score(interaction_design, sales),
         indicator_model.score(indicator_design, sales)))

print("interaction form:", {n: round(float(v), 3)
                            for n, v in zip(interaction_design.columns,
                                            interaction_model.coef_)},
      " intercept %.3f" % interaction_model.intercept_)
print("indicator form  :", {n: round(float(v), 3)
                            for n, v in zip(indicator_design.columns,
                                            indicator_model.coef_)},
      " intercept %.3f" % indicator_model.intercept_)

**The predictions differ by 6.8e-13 - floating-point noise - and the R-squared values agree to ten
decimal places. The coefficients look nothing alike.**

**Why they are the same model:** both designs span the same space. With two binary variables there are
exactly four cells, and any function of them is four numbers. The interaction form writes those four
numbers as `intercept`, `+promo`, `+weekend`, `+both`; the indicator form writes them as `intercept` plus
three offsets. One is an invertible linear map of the other, so the set of achievable predictions is
identical - and least squares finds the same best point in that set either way.

You can read the translation straight off:

- `cell_1` (weekend, no promo) = **38.046** = the `weekend` coefficient.
- `cell_2` (promo, no weekend) = **17.603** = the `promo` coefficient.
- `cell_3` (both) = **179.154** = 17.603 + 38.046 + 123.506, the sum of all three interaction terms.

**Three things follow, and they matter beyond this exercise.**

**A "significant interaction" is not a discovery about the world.** It is a statement about the
parameterisation you chose. The indicator form fits exactly the same data with no term called an
interaction at all.

**Coefficients are not the model.** Two people can report completely different numbers from the same fit
on the same data and both be right, which is why E16's advice was to report fitted differences rather
than coefficients.

**Parameterisation is still a real choice**, just not a statistical one. The interaction form scales to
continuous variables and to more than two factors; the indicator form does not, but it is unambiguous to
read and makes "what is the predicted value in this cell" a single number. Pick for the audience, and
know that the fit does not care.